# Bab 16. Studi Kasus: Dari Data Mentah ke Model

Kode pendamping buku *Python untuk Machine Learning dan Data
Science*. Jalankan selnya berurutan dari atas, sebab sebagian
sel memakai peubah dari sel sebelumnya.

Notebook ini dibangkitkan dari naskah buku. Jangan disunting di
sini, sunting listing pada berkas `.tex` lalu bangkitkan ulang.

## Persiapan

Bab ini melanjutkan contoh dari bab sebelumnya. Jalankan sel ini lebih dahulu supaya datanya tersedia.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import (LinearRegression,
    LogisticRegression, Ridge)
from sklearn.model_selection import (KFold, StratifiedKFold,
    train_test_split, cross_val_score, GridSearchCV,
    TimeSeriesSplit)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, roc_auc_score,
    f1_score)

from siapkan import kue_diperkaya

df = kue_diperkaya()

## 1. Pembagian berbasis waktu

In [ ]:
batas = pd.Timestamp("2025-12-01")
lat = df[df["tanggal"] < batas]
uji = df[df["tanggal"] >= batas]

print(len(lat), len(uji))

Keluaran yang diharapkan:

```
670 143
```

## 2. Model patokan

In [ ]:
from sklearn.dummy import DummyRegressor

dasar = DummyRegressor(strategy="median").fit(X_lat, y_lat)
p = dasar.predict(X_uji)
print(mean_absolute_error(y_uji, p), r2_score(y_uji, p))

Keluaran yang diharapkan:

```
5.26 -0.045
```

## 3. Menyusun prapemrosesan

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (OneHotEncoder,
                                   StandardScaler)

KAT = ["produk", "kategori", "kanal", "hari"]
NUM = ["harga", "bulan", "promo", "jelang_lebaran"]

pra = ColumnTransformer([
    ("kat", OneHotEncoder(handle_unknown="ignore"), KAT),
    ("num", StandardScaler(), NUM),
])

model = Pipeline([("pra", pra), ("m", Ridge(alpha=1.0))])

## 4. Melatih dengan sasaran log

In [ ]:
model.fit(X_lat, np.log1p(y_lat))
p = np.expm1(model.predict(X_uji))
p = np.clip(p, 1, None)     # jumlah minimal 1

## 5. Kepentingan permutasi

In [ ]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(model, X_uji, y_uji,
                             n_repeats=30, random_state=0,
                             scoring="r2")